In [ ]:
# %pip install requests numpy Pillow

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
from getpass import getpass
from pathlib import Path
import requests
import numpy as np

BASE_URL = "https://ai.healtricsapp.com"
API_KEY = os.getenv("ONENOTEAI_API_KEY") or getpass("Enter OneNoteAI API key manually (hidden): ")
if not API_KEY:
    raise ValueError("An API key is required.")

SESSION = requests.Session()
SESSION.headers.update({"X-API-Key": API_KEY}) #Attach the API key to every request because SESSION is used to send requests (eg post) so no need to readd that eash time you are sendinf a request
TIMEOUT = (10, 540)  # connection, read (seconds) #10 second: Maximum time allowed to establish the connection, 540 seconds: Maximum period of inactivity while waiting for response data (Models take time)
print("Gateway configured:", BASE_URL)

Gateway configured: https://ai.healtricsapp.com


In [7]:
# # First Test Example
# response = SESSION.post(
#     f"{BASE_URL}/embed",
#     json={"text": "Testing OneNoteAI from Jupyter"},
#     timeout=TIMEOUT,
# )

# response.raise_for_status()

# result = response.json()

# print("Model:", result["model"])
# print("Dimensions:", result["dimensions"])

An Example of what is sent as request
URL: https://ai.healtricsapp.com/embed
Method: POST
Body: {"text": "Hello"}
Header: X-API-Key: <your-secret>

In [2]:
#_post_json is used by  embed() and generate()
def _post_json(path: str, payload: dict) -> dict:
    try:
        response = SESSION.post(
            f"{BASE_URL}{path}", json=payload, timeout=TIMEOUT
        )
        response.raise_for_status()
        return response.json()
    
    except requests.HTTPError as exc:
        status = exc.response.status_code
        detail = exc.response.text[:300]
        raise RuntimeError(f"{path}: HTTP {status}: {detail}") from None
    except requests.RequestException as exc:
        raise RuntimeError(f"{path}: network request failed: {exc}") from exc

Reusable embed, generate and ocr functions using https://ai.healtricsapp.com endpoints /embed, /generate, /ocr

In [3]:
def embed(text: str) -> np.ndarray:
    #Get a 1D float embedding vector for one text string.
    result = _post_json("/embed", {"text": text})

    #Convert the response to a NumPy array
    vector = np.asarray(result["embedding"], dtype=np.float32)
    
    #The embedding must be one-dimensional (1D) and The embedding must not be empty
    if vector.ndim != 1 or vector.size == 0:
        raise ValueError("Unexpected embedding response shape")
    return vector

In [4]:
v1 = embed("The patient has high blood pressure.")
print(v1.shape)

(1024,)


In [5]:
def generate(prompt: str) -> str:
    #Generate text using the OneNoteAI Qwen vision/language model.
    result = _post_json("/generate", {"prompt": prompt})
    return result["answer"]

In [33]:
generate("Explain what is RAG used for in AI")

'**RAG** stands for **Retrieval-Augmented Generation**. It is a technique used in AI, particularly in large language models (LLMs), to improve the accuracy, relevance, and reliability of generated responses by combining **retrieval** of relevant information from a knowledge base with **generation** of responses using the model.\n\n---\n\n### 🎯 What RAG is Used For:\n\nRAG is designed to **enhance the performance of LLMs** by:\n\n1. **Reducing hallucinations** — LLMs can sometimes generate false or misleading information. RAG helps by grounding responses in real, retrieved data.\n2. **Improving factual accuracy** — Instead of relying solely on the model’s internal knowledge (which may be outdated or incomplete), RAG pulls in up-to-date or domain-specific information.\n3. **Supporting domain-specific or enterprise knowledge** — RAG can be trained to retrieve information from internal documents, databases, or knowledge bases, making it ideal for applications like customer support, legal r

In [34]:
generate("EExplain Azure Managed Identity in two sentences.")

'Azure Managed Identity is a service that automatically authenticates your Azure resources (like VMs or apps) to access Azure resources and other services without needing to manage passwords or credentials. It enables seamless, secure access to resources using Azure AD, reducing the risk of credential exposure and simplifying identity management.'

In [18]:
def ocr2(image_path: str | Path) -> str:
    #Upload PNG/JPEG/WEBP and return extracted text.

    #Convert the input into a Path object so you can use properties like .name and .suffix (.png)
    image_path = Path(image_path)

    #check for image extension (.png)/mime (image/png)
    mime = {".png": "image/png", ".jpg": "image/jpeg", ".jpeg": "image/jpeg", ".webp": "image/webp"}.get(image_path.suffix.lower())
    if not mime:
        raise ValueError("Use a PNG, JPEG, or WEBP image")
    
    #image_path.stat().st_size returns the file size in bytes : not >5mb
    if image_path.stat().st_size > 5 * 1024 * 1024:
        raise ValueError("Image exceeds the API's 5 MiB limit")
    
    #Open the image in binary mode ("rb" means read binary) and send OCR request. NB: OCR sends an image file, not just a JSON object.
    with image_path.open("rb") as handle:
        response = SESSION.post(
            f"{BASE_URL}/ocr",
            files={"file": (image_path.name, handle, mime)},
            timeout=TIMEOUT,
        )
    if not response.ok:
        raise RuntimeError(f"/ocr: HTTP {response.status_code}: {response.text[:300]}")
    #parses the JSON response and returns only the extracted text
    return response.json()["extracted_text"]

In [26]:
from pathlib import Path
from PIL import Image, ImageOps
from io import BytesIO


def optimize_image(image_path: str | Path, max_dimension: int = 1200) -> tuple[bytes, str]:
    """
    Resize an image if necessary and return its optimized
    bytes and MIME type, ready for upload.

    The original image is never modified.
    """

    image_path = Path(image_path)

    supported_types = {
        ".png": "image/png",
        ".jpg": "image/jpeg",
        ".jpeg": "image/jpeg",
        ".webp": "image/webp",
    }

    if image_path.suffix.lower() not in supported_types:
        raise ValueError("Use a PNG, JPEG, or WEBP image")

    with Image.open(image_path) as img:
        # Respect camera orientation metadata.
        img = ImageOps.exif_transpose(img)

        # Convert to RGB for consistent encoding.
        img = img.convert("RGB")

        # Resize only when the image exceeds the limit.
        img.thumbnail(
            (max_dimension, max_dimension),
            Image.Resampling.LANCZOS
        )

        # Save optimized image in memory as JPEG.
        buffer = BytesIO()
        img.save(buffer, format="JPEG", quality=90, optimize=True)

    return buffer.getvalue(), "image/jpeg"

In [24]:
def ocr(image_path: str | Path, max_dimension: int = 1000) -> str:
    
    #Optimize an image, upload it to OneNoteAI, and return the extracted text.
    

    image_path = Path(image_path)

    # Optimize the image before uploading.
    image_bytes, mime = optimize_image(
        image_path,
        max_dimension=max_dimension
    )

    # Enforce the API's 5 MiB upload limit.
    if len(image_bytes) > 5 * 1024 * 1024:
        raise ValueError("Optimized image exceeds the API's 5 MiB limit")

    # Upload the optimized image.
    response = SESSION.post(
        f"{BASE_URL}/ocr",
        files={
            "file": (
                f"{image_path.stem}_optimized.jpg",
                image_bytes,
                mime
            )
        },
        timeout=TIMEOUT,
    )

    if not response.ok:
        raise RuntimeError(
            f"/ocr: HTTP {response.status_code}: {response.text[:300]}"
        )
     #parses the JSON response and returns only the extracted text
    return response.json()["extracted_text"]

In [22]:
import time

image = "handwritten note 2.png"

for dimension in [1400, 1000, 800]:
    start = time.perf_counter()

    text = ocr(image, max_dimension=dimension)

    elapsed = time.perf_counter() - start

    print(f"\nMax dimension: {dimension}px")
    print(f"OCR time: {elapsed:.2f} seconds")
    print(f"Extracted characters: {len(text)}")


Max dimension: 1400px
OCR time: 174.14 seconds
Extracted characters: 1318

Max dimension: 1000px
OCR time: 108.08 seconds
Extracted characters: 1336

Max dimension: 800px
OCR time: 87.57 seconds
Extracted characters: 1348


In [23]:
import time

start = time.perf_counter()

text = ocr("Handwritten Clinical Notes and Study Form-1.png")

elapsed = time.perf_counter() - start

print(f"OCR processing time: {elapsed:.2f} seconds")
print("\nExtracted text:\n")
print(text)

OCR processing time: 158.99 seconds

Extracted text:

Patient Visit Note
10/17/2026

Name: John A. Mensah
Age: 57 yrs  Sex: M
MRN: 7845623

CC: Follow up for HTN & DM

HPI: Pt. returns today for routine f/u.
Reports BP has been "a bit high" at home.
No CP, no SOB. Occasional headache.
Sugars still up and down (150-220).
Admits to missing meds 1-2x/week.

Meds: Amlodipine 10mg daily
Metformin 500mg BID
Atorvastatin 20mg nightly

Exam: BP 148/92, HR 82, WT 94kg
Chest: clear  CVS: RRR   Ext: no edema

Plan: 1) Reinforce med adherence
2) Increase Metformin to 1000mg BID
3) Home BP & glucose log
4) F/u in 4 weeks

Dr. E. Okafor

Research Study - Enrollment Form

Study Title: AI-assisted review of clinic notes
IRB #: 2026-1187

Participant Information

Name: Grace N. Udo
DOB: 05/14/1988    Age: 38
Phone: (903) 555-8712
Email: grace.udo@gmail.com

Inclusion (V)
☑ Age ≥ 18
☑ Receiving care at this clinic
☑ Willing to share de-identified records

Exclusion ( )
☐ Pregnant
☐ Current participation

In [29]:
import time

start = time.perf_counter()

text = ocr("handwritten note 2.png")

elapsed = time.perf_counter() - start

print(f"OCR processing time: {elapsed:.2f} seconds")
print("\nExtracted text:\n")
print(text)

OCR processing time: 116.04 seconds

Extracted text:

Clinic Visit Note
11/10/2026

Patient: Michael A. Okafar
DOB: 04/17/1988 (Age 38)
MRN: 784563
Reason for visit: Follow-up (HTN +DM)

Subjective:
- Pt reports occasional headaches, especially
in the evening.
- No chest pain, no SOB.
- Admits to missing meds ~ 2X this week.
- Denies any new allergies.

Objective:
BR: 148/92 mmHg
HR: 86 bpm
Temp: 98.4°F
Wt: 91 kg (±200lb)

Assessment:
1) HTN - not at goal
2) Type 2 DM - slight improvement

Plan:
1) Continue Lisinopril 20 mg daily
2) Continue Metformin 1000mg BID
3) Reinforce medication adherence + diet
4) Follow-up in 3 months (02/10/2027)

Lab Results
Pt: Okafar, M.A.
MRN: 784563
Collected: 11/09/2026

Test        Result     Ref. Range   Flag
Glucose (Fasting)   132 mg/dL     70-99     H
A1c            7.9 %         4.0-5.6     H
Total Cholesterol   208 mg/dL    < 200     H
LDL            132 mg/dL    < 100     H
HDL            45 mg/dL     ≥ 40     N
Triglycerides    186 mg/dL    < 1

In [35]:
import time

start = time.perf_counter()

text = ocr("handwritten note 3.png")

elapsed = time.perf_counter() - start

print(f"OCR processing time: {elapsed:.2f} seconds")
print("\nExtracted text:\n")
print(text)

OCR processing time: 19.57 seconds

Extracted text:

Clinic. Note
10/17/2026

Patient: Mary E. Adams
Age: 45 yrs   Sex: F

CC: Cough and fever for 3 days.

HPI: Reports productive cough, temp up to 101.4°F,
body aches, and mild shortness of breath.

Exam: T 101.1°F, HR 96, BP 132/86, SpO2 97%
Lungs: mild wheeze, no crackles.

Plan: - Start Azithromycin 500mg day 1, then 250mg daily x 4 days.
- Acetaminophen 650mg q6h PRN fever.
- Rest, fluids, and follow up in 1 week.

Dr. K. Wilson
